# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vincentoei/flyrank-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Task type: Classification, with probability scoring used to produce a ranked list.

I am predicting a binary outcome: will a page grow in the next 30 days? The model outputs a probability for each page, and the final deliverable is a ranked list of pages sorted by growth probability. So the core task is classification, but the decision is a ranking decision.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Confirm the task type and the shape of the target
print("Task type: Classification (will the page grow in the next 30 days?)")
print("Output: probability of growth, used to rank pages for review.")
print("Final decision: a ranked list of top-K pages to expand, promote, or protect.")

Task type: Classification (will the page grow in the next 30 days?)
Output: probability of growth, used to rank pages for review.
Final decision: a ranked list of top-K pages to expand, promote, or protect.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target: growth_label_30d = 1 if the page's impressions in the next 30 days are at least 20% higher than in the prior 30 days, and the prior 30 days had at least 100 impressions.

This target is an observed future outcome, not a defined rule. It is measured after the decision point. The decision point is the last day of the prior 30-day window.

On the starter CSV, I cannot build a true future outcome because the data is a single snapshot. I will use a proxy target built from the current snapshot: impressions_last_30d >= 1.2 * impressions_prev_30d with impressions_prev_30d >= 100. This is only a stand-in. The real target will be built from the warehouse fact_content_daily_performance table using actual future dates.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parents[1]
df = pd.read_csv(ROOT / "data" / "raw" / "content_refresh_anonymized.csv")

# Proxy target on the starter snapshot (NOT a real future outcome)
df["growth_proxy"] = (
    (df["impressions_prev_30d"] >= 100)
    & (df["impressions_last_30d"] >= 1.20 * df["impressions_prev_30d"])
).astype(int)

print("Target definition (proxy on starter data):")
print("growth_proxy = 1 if impressions_last_30d >= 1.2 * impressions_prev_30d AND impressions_prev_30d >= 100")
print(f"\nProxy target distribution:\n{df['growth_proxy'].value_counts(normalize=True).round(3)}")
print(f"\nThis is a PROXY. The real target will be built from future dates in the warehouse.")

Target definition (proxy on starter data):
growth_proxy = 1 if impressions_last_30d >= 1.2 * impressions_prev_30d AND impressions_prev_30d >= 100

Proxy target distribution:
growth_proxy
0    0.922
1    0.078
Name: proportion, dtype: float64

This is a PROXY. The real target will be built from future dates in the warehouse.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Primary metric: Precision@50.

The reviewer will inspect the top 50 pages the model recommends. Precision@50 asks: how many of those 50 actually grew? This matches the real decision.

Secondary metrics: ROC-AUC and Average Precision. These measure how well the model separates growers from non-growers across the whole list.

A "good" result means the model meaningfully beats the baseline. For example, if the baseline Precision@50 is 0.10 and the model reaches 0.30, the reviewer gets 3x more value from the same 50 inspections.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.metrics import precision_score, roc_auc_score, average_precision_score

# Baseline: a simple rule that predicts growth if last-30d > prev-30d
df["baseline_rule"] = (df["impressions_last_30d"] > df["impressions_prev_30d"]).astype(int)

# Evaluate a simple rule as a baseline
y_true = df["growth_proxy"]
y_pred = df["baseline_rule"]

# Precision@50
def precision_at_k(y_true, scores, k=50):
    frame = pd.DataFrame({"y": y_true, "score": scores})
    top = frame.sort_values("score", ascending=False).head(k)
    return float(top["y"].mean())

baseline_p50 = precision_at_k(y_true, df["baseline_rule"], k=50)

print(f"Baseline Precision@50: {baseline_p50:.3f}")
print(f"Baseline ROC-AUC: {roc_auc_score(y_true, y_pred):.3f}")
print(f"Baseline Average Precision: {average_precision_score(y_true, y_pred):.3f}")
print("\nA good model will beat this baseline on Precision@50.")

Baseline Precision@50: 0.200
Baseline ROC-AUC: 0.885
Baseline Average Precision: 0.271

A good model will beat this baseline on Precision@50.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Unit of analysis: one row = one content item (one page).

The dataframe below shows the unit of analysis: each row is a unique page, with its content metadata, search performance, and the proxy target. The target column is a binary 0/1 flag.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Show the unit of analysis as a real dataframe
unit_cols = [
    "content_id",
    "client_id",
    "content_type",
    "content_age_days",
    "impressions_90d",
    "impressions_prev_30d",
    "impressions_last_30d",
    "avg_position",
    "ctr",
    "growth_proxy",
]

unit_df = df[unit_cols].copy()
print(f"Rows: {len(unit_df):,}")
print(f"Unique content_ids: {unit_df['content_id'].nunique():,}")
print(f"One row = one content item (page)")
print(f"\nFirst 5 rows:")
print(unit_df.head())

print("\nTarget column preview:")
print(unit_df[["content_id", "impressions_prev_30d", "impressions_last_30d", "growth_proxy"]].head(10))

Rows: 30,000
Unique content_ids: 30,000
One row = one content item (page)

First 5 rows:
             content_id          client_id     content_type  content_age_days  \
0  content_304f48230142  client_f369cb89fc  keyword article               187   
1  content_a1fb4e703a9e  client_4e07408562  keyword article               445   
2  content_9aa793d4d895  client_7f2253d7e2  keyword article               141   
3  content_331d6c4de07b  client_19581e27de  keyword article               463   
4  content_d99b7a2d90ca  client_3fdba35f04  keyword article               263   

   impressions_90d  impressions_prev_30d  impressions_last_30d  avg_position  \
0             3803                   987                   578          10.6   
1            15320                  5915                  2501          20.3   
2            12581                  6089                  2382          36.5   
3            11751                  4206                  3626           6.2   
4            19140      

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule like "pages where last-30d impressions grew vs prev-30d will keep growing" only uses one signal. Growth is messier. It depends on position, content age, freshness, CTR, engagement, content type, and keyword demand all at once.

ML wins when the pattern is real but tangled: many weak signals combine in non-linear ways. A model can weigh the signals differently for different pages. A fixed rule cannot.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Show how a simple rule misses pages that have other growth signals
simple_rule = df["impressions_last_30d"] > df["impressions_prev_30d"]

# Pages that do NOT pass the simple rule but DO pass the proxy target
missed = df[(~simple_rule) & (df["growth_proxy"] == 1)]
print(f"Pages the simple rule missed but proxy says grew: {len(missed):,}")

# Pages that pass the simple rule but do NOT pass the proxy target
false_alarms = df[(simple_rule) & (df["growth_proxy"] == 0)]
print(f"Pages the simple rule flagged but proxy says did not grow: {len(false_alarms):,}")

print(f"\nSimple rule accuracy: {(simple_rule == df['growth_proxy']).mean():.3f}")
print("A model can use position, age, CTR, engagement, and content type to reduce both missed pages and false alarms.")

Pages the simple rule missed but proxy says grew: 0
Pages the simple rule flagged but proxy says did not grow: 6,336

Simple rule accuracy: 0.789
A model can use position, age, CTR, engagement, and content type to reduce both missed pages and false alarms.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.